# Power BI Usage Intelligence: Forecasting Baseline

This notebook is the forecasting step in the project pipeline. It reads the
canonical narrow daily series produced by Notebook 04, validates it, runs
data-sufficiency checks, fits SARIMA and baseline models, selects the best
per-report model, and saves forecast and metric outputs.

**All logic lives in `src/`.  This notebook only orchestrates calls and
presents results.**

Pipeline position: `Notebook 04 (feature engineering)` → **Notebook 05 (forecasting)** → `Streamlit app`

## Run Checklist

1. Run `notebooks/04_feature_engineering.ipynb` to produce `data/processed/mart_report_daily_series.csv`.
2. Confirm the file exists at that path before running this notebook.
3. Run cells top-to-bottom in a clean kernel.

If `mart_report_daily_series.csv` is absent, the load cell raises `FileNotFoundError` with
instructions for which upstream step to run — do **not** generate synthetic replacement
data inside this notebook.

In [ ]:
import uuid
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.pipelines.run_forecasting_pipeline import (
    # constants
    DATE_COL,
    REPORT_ID_COL,
    REPORT_NAME_COL,
    VIEWS_COL,
    STANDARD_DATE_COL,
    STANDARD_REPORT_ID_COL,
    STANDARD_TARGET_COL,
    FORECAST_HORIZON_DAYS,
    MIN_DAYS,
    MIN_NONZERO_DAYS,
    MIN_NONZERO_RATIO,
    MIN_TOTAL_VIEWS,
    MODEL_NAME,
    EXPECTED_FORECAST_COLUMNS,
    # loading & validation
    load_canonical_daily_series,
    validate_forecasting_series_input,
    adapt_to_forecasting_schema,
    load_forecast_feature_input,
    # data-quality reporting
    run_data_quality_checks,
    # series building & eligibility
    build_daily_series_for_all_reports,
    filter_by_data_criteria,
    # forecasting
    run_forecasts_for_reports,
    # output
    save_latest_outputs,
    append_forecasts_history,
    append_metrics_history,
    update_realized_errors,
    get_project_root,
)

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
PROJECT_ROOT = get_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FORECAST_OUTPUT_DIR = OUTPUT_DIR / "forecasts"
METRICS_OUTPUT_DIR = OUTPUT_DIR / "metrics"

FORECAST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional: restrict forecasting to a subset of report names.
# Set to set() to run all reports.
ALLOWED_REPORT_NAMES: set = set()

RUN_TIMESTAMP = pd.Timestamp.now()
RUN_ID = RUN_TIMESTAMP.strftime("%Y%m%d_%H%M%S") + "_" + str(uuid.uuid4())[:8]

print(f"RUN_ID:           {RUN_ID}")
print(f"RUN_TIMESTAMP:    {RUN_TIMESTAMP}")
print(f"PROJECT_ROOT:     {PROJECT_ROOT.resolve()}")
print(f"PROCESSED_DIR:    {PROCESSED_DIR.resolve()}")
print(f"FORECAST_DIR:     {FORECAST_OUTPUT_DIR.resolve()}")
print(f"METRICS_DIR:      {METRICS_OUTPUT_DIR.resolve()}")

## 1) Load and Validate the Daily Report Series

`load_forecast_feature_input` loads **only** `mart_report_daily_series.csv`
via `load_canonical_daily_series`.  It raises `FileNotFoundError` if that file
is absent — no fallback, no synthetic generation.  `validate_forecasting_series_input`
enforces the full structural contract (no nulls, no gaps, no negatives, integer-valued views).

In [ ]:
# Raises FileNotFoundError if mart_report_daily_series.csv is absent.
daily_series_df, report_views_df, active_series_input_file = load_forecast_feature_input(PROJECT_ROOT)

print(f"Loaded: {active_series_input_file.relative_to(PROJECT_ROOT)}")
print(f"Shape:  {daily_series_df.shape}")
print(f"Reports: {daily_series_df[STANDARD_REPORT_ID_COL].nunique()}")
print(f"Date range: {daily_series_df[STANDARD_DATE_COL].min().date()} -> {daily_series_df[STANDARD_DATE_COL].max().date()}")
daily_series_df.head()

In [ ]:
# EDA snapshot — total views and row count per report
summary = (
    report_views_df.groupby(REPORT_NAME_COL)[VIEWS_COL]
    .agg(total_views="sum", row_count="count")
    .sort_values("total_views", ascending=False)
)
summary.head(10)

In [ ]:
# Plot the first report's daily usage series
sample_report = report_views_df[REPORT_NAME_COL].iloc[0]
daily_sample = (
    report_views_df[report_views_df[REPORT_NAME_COL] == sample_report]
    .groupby(DATE_COL)[VIEWS_COL]
    .sum()
)

plt.figure(figsize=(12, 4))
plt.plot(daily_sample.index, daily_sample.values)
plt.title(f"Sample Daily Usage: {sample_report}")
plt.xlabel("Date")
plt.ylabel("Views")
plt.tight_layout()
plt.show()

## 2) Data Quality Checks

`run_data_quality_checks` returns one row per structural check on the canonical
`[date, report_id, daily_views]` frame.  `validate_forecasting_series_input`
already ran inside `load_forecast_feature_input` and would have raised on any
structural failure; this table provides a human-readable summary.

In [ ]:
dq = run_data_quality_checks(daily_series_df)
dq

## 3) Build Per-Report Series and Apply Data Sufficiency Gates

`build_daily_series_for_all_reports` splits the validated frame into one
`pd.Series` per report.  `filter_by_data_criteria` then gates on:

| Criterion | Threshold |
|---|---|
| Minimum calendar days | 90 |
| Minimum nonzero-view days | 35 |
| Minimum nonzero ratio | 0.25 |
| Minimum total views | 120 |

The forecasting layer never fills gaps — the canonical mart owns zero-fill.

In [ ]:
series_dict, name_lookup, provenance = build_daily_series_for_all_reports(report_views_df)
passing_ids, data_diag = filter_by_data_criteria(series_dict, provenance)

if ALLOWED_REPORT_NAMES:
    passing_ids = [r for r in passing_ids if name_lookup.get(r) in ALLOWED_REPORT_NAMES]

print(f"Total reports in series: {len(series_dict)}")
print(f"Passing data criteria:   {len(passing_ids)}")
data_diag.sort_values("passes_data_criteria", ascending=False).head(10)

## 4) Run Forecasts

`run_forecasts_for_reports` fits SARIMA (via `pmdarima.auto_arima`) and two
baseline models (naive last-value, seasonal naive) for every report that passed
data-sufficiency gates.  It selects the best model by MAE and applies reliability
checks.  Only forecasts from reliable models are published to the output files.

In [ ]:
forecast_table, metrics_table, data_diag, model_comparison_table = run_forecasts_for_reports(
    report_views_df,
    run_id=RUN_ID,
    run_timestamp=RUN_TIMESTAMP,
    horizon_days=FORECAST_HORIZON_DAYS,
)

print(f"Reports processed:             {len(metrics_table)}")
print(f"Forecast rows published:       {len(forecast_table)}")
print(f"Model comparison rows:         {len(model_comparison_table)}")
print(f"Reports passing data criteria: {int(data_diag['passes_data_criteria'].sum())}/{len(data_diag)}")
metrics_table.head()

## 5) Save Outputs

In [ ]:
latest_forecast_file, latest_metrics_file, latest_model_comparison_file = save_latest_outputs(
    forecast_table,
    metrics_table,
    model_comparison_table,
    project_root=PROJECT_ROOT,
    run_id=RUN_ID,
    run_timestamp=RUN_TIMESTAMP,
)
append_forecasts_history(forecast_table, project_root=PROJECT_ROOT)
append_metrics_history(
    metrics_table,
    project_root=PROJECT_ROOT,
    run_id=RUN_ID,
    run_timestamp=RUN_TIMESTAMP,
)
realized_errors_df = update_realized_errors(report_views_df, project_root=PROJECT_ROOT)

print(f"Latest forecasts  -> {latest_forecast_file.relative_to(PROJECT_ROOT)}")
print(f"Latest metrics    -> {latest_metrics_file.relative_to(PROJECT_ROOT)}")
print(f"Realized errors rows: {len(realized_errors_df)}")

## 6) Model Comparison and Forecast Review

In [ ]:
# Per-report reliability summary
if not metrics_table.empty:
    display_cols = [
        c for c in [
            "report_name", "passes_data_criteria", "selected_model", "selected_mae",
            "selected_rmse", "selected_wape", "forecast_reliable",
            "passes_wape_reliability", "passes_zero_share",
            "mae_arima", "mae_naive", "mae_seasonal_naive", "model_str",
        ]
        if c in metrics_table.columns
    ]
    display(
        metrics_table[display_cols]
        .sort_values(["forecast_reliable", "selected_mae"], ascending=[False, True])
        .head(15)
    )

In [ ]:
# Per-report, per-model comparison
if not model_comparison_table.empty:
    display_cols = [
        c for c in [
            "report_id", "report_name", "model_name",
            "mae", "rmse", "wape", "selected_model_flag", "forecast_reliable",
        ]
        if c in model_comparison_table.columns
    ]
    display(
        model_comparison_table[display_cols]
        .sort_values(["report_name", "selected_model_flag", "mae"], ascending=[True, False, True])
        .head(30)
    )

## 7) Forecast Visualisation

Plot the selected model's test-period fitted values and future forecast for a
sample reliable report.

In [ ]:
if not forecast_table.empty and not metrics_table.empty:
    reliable = metrics_table[metrics_table.get("forecast_reliable", pd.Series(dtype=bool)) == True]
    if reliable.empty:
        print("No reliable forecasts to plot.")
    else:
        sample_rid = reliable.iloc[0]["report_id"] if "report_id" in reliable.columns else None
        if sample_rid is not None:
            sample_name = name_lookup.get(str(sample_rid), str(sample_rid))
            sample_fc = forecast_table[forecast_table["ReportId"] == sample_rid].copy()
            sample_fc[DATE_COL] = pd.to_datetime(sample_fc[DATE_COL])

            actual_part = sample_fc[sample_fc["IsForecast"] == 0]
            forecast_part = sample_fc[sample_fc["IsForecast"] == 1]

            plt.figure(figsize=(14, 5))
            if not actual_part.empty:
                plt.plot(actual_part[DATE_COL], actual_part["actual"], label="Actual (test)", color="steelblue")
                plt.plot(actual_part[DATE_COL], actual_part["forecast"], label="Fitted (test)", color="orange", linestyle="--")
            if not forecast_part.empty:
                plt.plot(forecast_part[DATE_COL], forecast_part["forecast"], label="Forecast (future)", color="green", linestyle="--")
                if "lower_ci" in forecast_part.columns and "upper_ci" in forecast_part.columns:
                    plt.fill_between(
                        forecast_part[DATE_COL],
                        forecast_part["lower_ci"].clip(0),
                        forecast_part["upper_ci"],
                        alpha=0.2,
                        color="green",
                        label="95% CI",
                    )
            plt.title(f"Forecast: {sample_name}")
            plt.xlabel("Date")
            plt.ylabel("Daily Views")
            plt.legend()
            plt.tight_layout()
            plt.show()

## 8) Next Steps

- **Rolling-origin backtesting** (not yet implemented): evaluate models on multiple
  historical cut-off dates rather than a single train/test split.
- **Insight generation**: pipe `forecast_table` + `metrics_table` into
  `src/genai/insight_generator.py` to produce per-report GenAI summaries.
- **Streamlit app**: outputs in `outputs/forecasts/` and `outputs/metrics/` are
  consumed directly by `src/app/streamlit_app.py`.